# 04 - Training the FC-Siam-Conc Model

## Objectives

The objective of this notebook is to train the FC-Siam-Conc model on the LEVIR-CD dataset for binary change detection.

The specific objectives are:

- Define the loss function.
- Configure the optimizer.
- Implement the training loop.
- Implement the validation loop.
- Save the best-performing model.
- Visualize training and validation loss curves.
- Prepare the trained model for evaluation.

In [1]:
import os
import sys

project_root = os.path.abspath("..")
sys.path.insert(0, project_root)

print(project_root)

C:\Users\balas\capstone


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm

import matplotlib.pyplot as plt
from src.models.fc_siam_conc import FCSiamConc

In [3]:
LEARNING_RATE = 1e-4
BATCH_SIZE = 4
NUM_EPOCHS = 30
NUM_WORKERS = 2

IMAGE_SIZE = 1024

In [4]:
import os

SAVE_DIR = "../saved_models/fc_siam_conc"

os.makedirs(SAVE_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(SAVE_DIR, "best_model.pth")
LAST_MODEL_PATH = os.path.join(SAVE_DIR, "last_model.pth")

In [5]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using:", DEVICE)

Using: cuda


In [6]:
model = FCSiamConc().to(DEVICE)

print(model.__class__.__name__)

FCSiamConc


In [7]:
TRAIN_PATH = "../data/train"
VAL_PATH = "../data/val"
TEST_PATH = "../data/test"

In [8]:
from src.datasets.levir_dataset import (
    LEVIRDataset,
    train_transform,
    val_transform
)

train_dataset = LEVIRDataset(
    TRAIN_PATH,
    transform=train_transform
)

val_dataset = LEVIRDataset(
    VAL_PATH,
    transform=val_transform
)

test_dataset = LEVIRDataset(
    TEST_PATH,
    transform=val_transform
)

In [9]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [10]:
print(f"Training Samples   : {len(train_dataset)}")
print(f"Validation Samples : {len(val_dataset)}")
print(f"Testing Samples    : {len(test_dataset)}")

Training Samples   : 445
Validation Samples : 64
Testing Samples    : 128


In [11]:
before, after, mask = next(iter(train_loader))

print("Before :", before.shape)
print("After  :", after.shape)
print("Mask   :", mask.shape)

Before : torch.Size([4, 3, 256, 256])
After  : torch.Size([4, 3, 256, 256])
Mask   : torch.Size([4, 256, 256])


# Loss Function

Since the LEVIR-CD dataset contains a significant class imbalance
(approximately 4.59% changed pixels), using only Binary Cross Entropy
may bias the model toward predicting unchanged pixels.

To address this issue, a hybrid BCE + Dice Loss is used.

• BCE provides stable pixel-wise supervision.
• Dice Loss improves segmentation overlap.

The combination has become a standard choice for binary change detection.

In [12]:
import torch
import torch.nn as nn


class DiceLoss(nn.Module):

    def __init__(self, smooth=1):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):

        probs = torch.sigmoid(logits)

        probs = probs.view(-1)
        targets = targets.view(-1)

        intersection = (probs * targets).sum()

        dice = (
            2 * intersection + self.smooth
        ) / (
            probs.sum() + targets.sum() + self.smooth
        )

        return 1 - dice


class BCEDiceLoss(nn.Module):

    def __init__(self):
        super().__init__()

        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, logits, targets):

        bce = self.bce(
            logits.squeeze(1),
            targets
        )

        dice = self.dice(
            logits,
            targets
        )

        return bce + dice

# Optimizer

The Adam optimizer is selected because it adapts learning rates
for each parameter individually, resulting in faster convergence
for deep convolutional neural networks.

In [13]:
criterion = BCEDiceLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

# Learning Rate Scheduler

ReduceLROnPlateau monitors the validation loss.

If the validation loss stops improving,
the learning rate is automatically reduced
to improve convergence.

In [14]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

# Evaluation Metrics

To evaluate the performance of the FC-Siam-Conc model, the following metrics are used:

- Binary Cross Entropy + Dice Loss (Training Objective)
- Intersection over Union (IoU)
- Dice Score
- Precision
- Recall
- F1 Score

These metrics provide a comprehensive evaluation of binary change detection performance.

In [15]:
import torch


def calculate_metrics(logits, targets):

    probs = torch.sigmoid(logits)

    preds = (probs > 0.5).float()

    preds = preds.squeeze(1)

    targets = targets.float()

    tp = (preds * targets).sum()

    fp = (preds * (1 - targets)).sum()

    fn = ((1 - preds) * targets).sum()

    tn = ((1 - preds) * (1 - targets)).sum()

    smooth = 1e-6

    precision = (tp + smooth) / (tp + fp + smooth)

    recall = (tp + smooth) / (tp + fn + smooth)

    f1 = (2 * precision * recall) / (precision + recall + smooth)

    iou = (tp + smooth) / (tp + fp + fn + smooth)

    dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)

    accuracy = (tp + tn) / (tp + tn + fp + fn + smooth)

    return {
        "accuracy": accuracy.item(),
        "precision": precision.item(),
        "recall": recall.item(),
        "f1": f1.item(),
        "iou": iou.item(),
        "dice": dice.item()
    }

In [16]:
before, after, mask = next(iter(train_loader))

before = before.to(DEVICE)
after = after.to(DEVICE)
mask = mask.to(DEVICE)

model.eval()

with torch.no_grad():

    outputs = model(before, after)

metrics = calculate_metrics(outputs, mask)

metrics

{'accuracy': 0.04871368408203125,
 'precision': 0.04871368408203125,
 'recall': 1.0,
 'f1': 0.09290169924497604,
 'iou': 0.04871368408203125,
 'dice': 0.09290178120136261}

In [17]:
before, after, mask = next(iter(train_loader))

print(before.shape)
print(after.shape)
print(mask.shape)

torch.Size([4, 3, 256, 256])
torch.Size([4, 3, 256, 256])
torch.Size([4, 256, 256])


In [18]:
before = before.to(DEVICE)
after = after.to(DEVICE)

model.eval()

with torch.no_grad():
    outputs = model(before, after)

print(outputs.shape)

torch.Size([4, 1, 256, 256])


In [19]:
mask = mask.to(DEVICE)

loss = criterion(outputs, mask)

print(loss.item())

1.6133326292037964


# Model Training

The FC-Siam-Conc model is trained using mini-batch gradient descent.

Training workflow:

1. Forward Pass
2. Compute BCE + Dice Loss
3. Backpropagation
4. Update Parameters
5. Evaluate on Validation Set
6. Save Best Model

In [20]:
from tqdm.auto import tqdm

In [22]:
train_losses = []
val_losses = []

best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):

    ###################################
    # TRAINING
    ###################################

    model.train()

    train_loss = 0

    train_metrics = {
        "iou": 0,
        "dice": 0,
        "f1": 0
    }

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"
    )

    for before, after, mask in train_bar:

        before = before.to(DEVICE)
        after = after.to(DEVICE)
        mask = mask.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(before, after)

        loss = criterion(outputs, mask)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        metrics = calculate_metrics(outputs, mask)

        train_metrics["iou"] += metrics["iou"]
        train_metrics["dice"] += metrics["dice"]
        train_metrics["f1"] += metrics["f1"]

        train_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss /= len(train_loader)
    train_losses.append(train_loss)

    train_iou = train_metrics["iou"] / len(train_loader)
    train_dice = train_metrics["dice"] / len(train_loader)
    train_f1 = train_metrics["f1"] / len(train_loader)

    ###################################
    # VALIDATION
    ###################################

    model.eval()

    val_loss = 0

    val_metrics = {
        "iou": 0,
        "dice": 0,
        "f1": 0
    }

    with torch.no_grad():

        for before, after, mask in val_loader:

            before = before.to(DEVICE)
            after = after.to(DEVICE)
            mask = mask.to(DEVICE)

            outputs = model(before, after)

            loss = criterion(outputs, mask)

            val_loss += loss.item()

            metrics = calculate_metrics(outputs, mask)

            val_metrics["iou"] += metrics["iou"]
            val_metrics["dice"] += metrics["dice"]
            val_metrics["f1"] += metrics["f1"]

    val_loss /= len(val_loader)
    val_losses.append(val_loss)

    val_iou = val_metrics["iou"] / len(val_loader)
    val_dice = val_metrics["dice"] / len(val_loader)
    val_f1 = val_metrics["f1"] / len(val_loader)

    ###################################
    # Learning Rate Scheduler
    ###################################

    scheduler.step(val_loss)

    ###################################
    # Save Best Model
    ###################################

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print("\n✅ Best model updated!")

    ###################################
    # Epoch Summary
    ###################################

    print("\n" + "="*60)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")

    print("="*60)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")

    print()

    print(f"Train IoU  : {train_iou:.4f}")
    print(f"Val IoU    : {val_iou:.4f}")

    print()

    print(f"Train Dice : {train_dice:.4f}")
    print(f"Val Dice   : {val_dice:.4f}")

    print()

    print(f"Train F1   : {train_f1:.4f}")
    print(f"Val F1     : {val_f1:.4f}")

###################################
# Save Last Model
###################################

torch.save(
    model.state_dict(),
    LAST_MODEL_PATH
)

print("\n" + "="*60)
print("🎉 Training Completed Successfully!")
print("="*60)
print(f"Best Model : {BEST_MODEL_PATH}")
print(f"Last Model : {LAST_MODEL_PATH}")

Epoch 1/30:   0%|          | 0/112 [00:00<?, ?it/s]


✅ Best model updated!

Epoch 1/30
Train Loss : 1.1121
Val Loss   : 1.0646

Train IoU  : 0.4791
Val IoU    : 0.5338

Train Dice : 0.6372
Val Dice   : 0.6916

Train F1   : 0.6372
Val F1     : 0.6916


Epoch 2/30:   0%|          | 0/112 [00:00<?, ?it/s]


✅ Best model updated!

Epoch 2/30
Train Loss : 1.0228
Val Loss   : 0.9902

Train IoU  : 0.5122
Val IoU    : 0.5506

Train Dice : 0.6611
Val Dice   : 0.7050

Train F1   : 0.6611
Val F1     : 0.7050


Epoch 3/30:   0%|          | 0/112 [00:00<?, ?it/s]

...


✅ Best model updated!

Epoch 28/30
Train Loss : 0.1657
Val Loss   : 0.2328

Train IoU  : 0.8169
Val IoU    : 0.7381

Train Dice : 0.8988
Val Dice   : 0.8465

Train F1   : 0.8988
Val F1     : 0.8465


Epoch 29/30:   0%|          | 0/112 [00:00<?, ?it/s]


✅ Best model updated!

Epoch 29/30
Train Loss : 0.1724
Val Loss   : 0.2259

Train IoU  : 0.8103
Val IoU    : 0.7397

Train Dice : 0.8936
Val Dice   : 0.8473

Train F1   : 0.8936
Val F1     : 0.8473


Epoch 30/30:   0%|          | 0/112 [00:00<?, ?it/s]


Epoch 30/30
Train Loss : 0.1752
Val Loss   : 0.2349

Train IoU  : 0.8050
Val IoU    : 0.7309

Train Dice : 0.8904
Val Dice   : 0.8415

Train F1   : 0.8904
Val F1     : 0.8415

🎉 Training Completed Successfully!
Best Model : ../saved_models/fc_siam_conc\best_model.pth
Last Model : ../saved_models/fc_siam_conc\last_model.pth
